# 04B — Advanced Retrieval

Building on `rag_fundamentals.ipynb`. Where that notebook established index → retrieve → generate, this one fixes the most common production failure: retrieval quality.

**Sections:**
1. The vocabulary mismatch problem
2. BM25 keyword search
3. Reciprocal Rank Fusion (hybrid search)
4. Cross-encoder re-ranking
5. HyDE — Hypothetical Document Embeddings
6. Query decomposition
7. The complete production pipeline
8. Key observations

In [ ]:
%pip install anthropic chromadb sentence-transformers rank-bm25

In [ ]:
import anthropic
import chromadb
import json
import os
import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

MODEL = "claude-sonnet-4-20250514"
client = anthropic.Anthropic()
embedder = SentenceTransformer("all-MiniLM-L6-v2")

print(f"Model: {MODEL}")
print(f"Embedder loaded: all-MiniLM-L6-v2")

---
## Section 1 — The Vocabulary Mismatch Problem

Financial text is full of abbreviations, acronyms, and formal vs informal vocabulary. A user asking about "Goldman" expects results from documents that say "GS Equity Research." Pure semantic search handles this imperfectly — the embedding spaces for formal and informal variants are close but not identical, and under competition from other documents the gap matters.

In [ ]:
# Mini corpus with deliberate vocabulary mismatches
corpus = [
    {
        "id": "doc_00",
        "text": "GS Equity Research initiates coverage on Global Semiconductors sector with Overweight. "
                "Supply constraints in leading-edge fabs continue to support pricing power through 2026."
    },
    {
        "id": "doc_01",
        "text": "NVIDIA Corporation reported Q1 FY2027 net income of $18.8B, up 72% YoY. "
                "Data center revenue reached $22.6B driven by Blackwell Ultra ramp."
    },
    {
        "id": "doc_02",
        "text": "GS Equity Research reiterates Buy on NVIDIA Corporation. Price target raised to $1,100. "
                "Blackwell demand is tracking above internal estimates across all hyperscaler accounts."
    },
    {
        "id": "doc_03",
        "text": "Portfolio equity allocation as of May 2026: Technology 74.2%, Financials 7.7%, Cash 6.3%. "
                "Concentration in large-cap growth has increased 8 percentage points YTD."
    },
    {
        "id": "doc_04",
        "text": "Monthly P&L summary: realized net income from closed positions $4,820. "
                "Unrealized gains across open positions $38,215. Best performer: NVIDIA Corporation +41%."
    },
    {
        "id": "doc_05",
        "text": "Global Semiconductors supply chain disruption risk remains elevated. "
                "TSMC advanced node capacity utilization at 98%. Lead times extending into Q3 2026."
    },
    {
        "id": "doc_06",
        "text": "Apple Inc. equity holding: 150 shares, avg cost $182.40, current value $31,762. "
                "Represents 11.1% of total portfolio equity allocation."
    },
    {
        "id": "doc_07",
        "text": "GS Equity Research macro outlook: Fed funds rate expected to decline 50bps in H2 2026. "
                "Duration risk in fixed income portfolios remains elevated at current yield levels."
    },
    {
        "id": "doc_08",
        "text": "JPMorgan Chase net income Q1 2026: $14.1B. Return on equity 17.2%. "
                "NII guidance revised downward on spread compression outlook."
    },
    {
        "id": "doc_09",
        "text": "Microsoft Corporation equity holding: 80 shares, avg cost $378.20, unrealized gain $3,768. "
                "Azure growth re-acceleration supports Overweight thesis from GS Equity Research."
    }
]

# Queries with vocabulary mismatch vs corpus
mismatch_queries = [
    {
        "q": "What did Goldman say about semiconductors?",
        "mismatch": "Goldman → GS Equity Research, semiconductors → Global Semiconductors",
        "relevant_ids": ["doc_00", "doc_02", "doc_05", "doc_07"]
    },
    {
        "q": "NVDA earnings this quarter",
        "mismatch": "NVDA → NVIDIA Corporation",
        "relevant_ids": ["doc_01", "doc_02", "doc_04"]
    },
    {
        "q": "How are my stock holdings allocated?",
        "mismatch": "stock holdings → equity allocation",
        "relevant_ids": ["doc_03", "doc_06", "doc_09"]
    }
]

# Embed the full corpus
corpus_texts = [d["text"] for d in corpus]
corpus_ids   = [d["id"]   for d in corpus]
corpus_embeddings = embedder.encode(corpus_texts, normalize_embeddings=True)

def semantic_search(query: str, top_k: int = 5):
    q_emb = embedder.encode([query], normalize_embeddings=True)[0]
    scores = corpus_embeddings @ q_emb
    ranked = sorted(zip(corpus_ids, scores.tolist()), key=lambda x: x[1], reverse=True)
    return ranked[:top_k]

print("=" * 72)
print("PURE SEMANTIC SEARCH — vocabulary mismatch queries")
print("=" * 72)
for item in mismatch_queries:
    results = semantic_search(item["q"], top_k=5)
    retrieved_ids = [r[0] for r in results]
    missed = [rid for rid in item["relevant_ids"] if rid not in retrieved_ids]
    print(f"\nQ: {item['q']}")
    print(f"   Mismatch : {item['mismatch']}")
    print(f"   Retrieved: {retrieved_ids}")
    print(f"   Relevant : {item['relevant_ids']}")
    print(f"   MISSED   : {missed if missed else 'none'}")

These are not edge cases. Financial text is full of abbreviations, acronyms, and formal vs informal vocabulary. "Goldman" and "GS Equity Research" are semantically close but not identical embeddings — under competition from other documents, that gap causes misses. Pure semantic search fails systematically in this domain.

---
## Section 2 — BM25 Keyword Search

BM25 is term-frequency based — it doesn't use embeddings at all. It finds exact and near-exact term matches. The flip side: it can't bridge vocabulary gaps. "Goldman" and "GS" are completely unrelated to BM25.

In [ ]:
# Build BM25 index
tokenized_corpus = [doc.lower().split() for doc in corpus_texts]
bm25_index = BM25Okapi(tokenized_corpus)

def bm25_search(query: str, top_k: int = 5):
    tokenized_query = query.lower().split()
    scores = bm25_index.get_scores(tokenized_query)
    ranked = sorted(zip(corpus_ids, scores.tolist()), key=lambda x: x[1], reverse=True)
    return ranked[:top_k]

print("=" * 72)
print("BM25 SEARCH — same vocabulary mismatch queries")
print("=" * 72)

comparison = []
for item in mismatch_queries:
    sem_results = semantic_search(item["q"], top_k=5)
    bm25_results = bm25_search(item["q"], top_k=5)
    sem_ids  = [r[0] for r in sem_results  if r[1] > 0.0]
    bm25_ids = [r[0] for r in bm25_results if r[1] > 0.0]
    unique_to_sem   = [i for i in sem_ids  if i not in bm25_ids]
    unique_to_bm25  = [i for i in bm25_ids if i not in sem_ids]
    comparison.append({
        "q": item["q"], "sem": sem_ids, "bm25": bm25_ids,
        "u_sem": unique_to_sem, "u_bm25": unique_to_bm25
    })
    print(f"\nQ: {item['q']}")
    print(f"   Semantic unique : {unique_to_sem}")
    print(f"   BM25 unique     : {unique_to_bm25}")

print("\n" + "=" * 72)
print(f"{'Query':<42} {'Sem hits':>8} {'BM25 hits':>9} {'Only sem':>8} {'Only BM25':>9}")
print("-" * 80)
for c in comparison:
    print(f"{c['q'][:41]:<42} {len(c['sem']):>8} {len(c['bm25']):>9} {len(c['u_sem']):>8} {len(c['u_bm25']):>9}")

BM25 finds "GS" if the query says "GS." It misses "GS" if the query says "Goldman." Neither method alone is complete. Each catches what the other misses — which is exactly the motivation for combining them.

---
## Section 3 — Reciprocal Rank Fusion

RRF merges two ranked lists without needing their scores to be on the same scale. It uses only rank position. The k=60 constant dampens the advantage of very high ranks — a rank-1 result doesn't dominate the fusion as aggressively as it would with raw score combination.

In [ ]:
def reciprocal_rank_fusion(
    semantic_results: list[tuple],
    bm25_results: list[tuple],
    k: int = 60
) -> list[tuple]:
    scores: dict = {}
    for rank, (doc_id, _score) in enumerate(semantic_results):
        scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank + 1)
    for rank, (doc_id, _score) in enumerate(bm25_results):
        scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


def hybrid_search(query: str, top_k: int = 5):
    sem  = semantic_search(query, top_k=len(corpus))
    bm25 = bm25_search(query,    top_k=len(corpus))
    fused = reciprocal_rank_fusion(sem, bm25)
    return fused[:top_k]


print("=" * 72)
print("HYBRID SEARCH (RRF) — vs semantic and BM25 alone")
print("=" * 72)

for item in mismatch_queries:
    sem_top3    = [r[0] for r in semantic_search(item["q"], top_k=3)]
    bm25_top3   = [r[0] for r in bm25_search(item["q"],    top_k=3) if r[1] > 0]
    hybrid_top3 = [r[0] for r in hybrid_search(item["q"],  top_k=3)]

    # Documents that appear in BOTH signal lists (natural RRF boost)
    in_both = [d for d in hybrid_top3 if d in sem_top3 and d in bm25_top3]

    # Documents recovered by hybrid that neither alone had in top 3
    recovered = [d for d in hybrid_top3 if d not in sem_top3 and d not in bm25_top3]

    print(f"\nQ: {item['q']}")
    print(f"   Semantic top-3  : {sem_top3}")
    print(f"   BM25 top-3      : {bm25_top3}")
    print(f"   Hybrid top-3    : {hybrid_top3}")
    print(f"   In both (boosted): {in_both}")
    print(f"   Recovered by RRF : {recovered if recovered else 'none'}")

RRF is elegant precisely because it ignores raw scores. Cosine similarity and BM25 scores are on completely different scales — combining them directly requires calibration. RRF sidesteps that entirely: rank position is the only signal. A document that appears in both lists gets contributions from both ranks, naturally surfacing to the top without any scale normalization.

---
## Section 4 — Cross-Encoder Re-ranking

Bi-encoders (like `all-MiniLM-L6-v2`) embed query and document independently — they never interact during scoring. Cross-encoders read query and document together as a single input. That joint attention signal is what makes reranking more precise. Too slow to run over a full corpus; essential for the top K.

In [ ]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


def rerank(query: str, doc_ids: list[str], top_n: int = 3) -> list[tuple]:
    id_to_text = {d["id"]: d["text"] for d in corpus}
    pairs = [(query, id_to_text[did]) for did in doc_ids]
    scores = cross_encoder.predict(pairs)
    ranked = sorted(zip(doc_ids, scores.tolist()), key=lambda x: x[1], reverse=True)
    return ranked[:top_n]


print("=" * 72)
print("HYBRID → CROSS-ENCODER RERANKING")
print("=" * 72)

for item in mismatch_queries:
    # Stage 1: hybrid search, retrieve top 10
    hybrid_10 = [r[0] for r in hybrid_search(item["q"], top_k=10)]

    # Stage 2: cross-encoder rerank to top 3
    reranked_3 = rerank(item["q"], hybrid_10, top_n=3)
    hybrid_3   = [r[0] for r in hybrid_search(item["q"], top_k=3)]
    reranked_ids = [r[0] for r in reranked_3]

    order_changed = hybrid_3 != reranked_ids

    print(f"\nQ: {item['q']}")
    print(f"   Hybrid top-3   : {hybrid_3}")
    print(f"   Reranked top-3 : {reranked_ids}")
    print(f"   Order changed  : {order_changed}")
    for doc_id, score in reranked_3:
        print(f"     {doc_id}  cross-encoder score: {score:.4f}")

Bi-encoders embed query and document separately — they never interact. Cross-encoders read them together as a pair: the query attends over the document and the document attends over the query simultaneously. That joint signal is what produces more precise relevance judgments. The cost trade-off is clear: cross-encoding 10,000 documents per query would be prohibitive; cross-encoding the top 10 candidates adds milliseconds.

---
## Section 5 — HyDE (Hypothetical Document Embeddings)

HyDE bridges the vocabulary gap by generating a hypothetical document that *would* answer the query, then using that document's embedding for retrieval instead of the query's embedding. The hypothesis is that a generated financial excerpt uses domain vocabulary that aligns better with indexed financial documents than the casual user query does.

In [ ]:
def cosine_similarity_1d(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10))


def hyde_retrieve(query: str, top_k: int = 3) -> tuple[list[tuple], str]:
    hypothetical = client.messages.create(
        model=MODEL,
        max_tokens=200,
        system=(
            "You are a financial analyst. Write a short excerpt from an analyst report "
            "or portfolio document that would directly answer this question. "
            "Use formal financial terminology. 2-3 sentences only."
        ),
        messages=[{"role": "user", "content": query}]
    ).content[0].text.strip()

    hyp_embedding = embedder.encode([hypothetical], normalize_embeddings=True)[0]
    scores = corpus_embeddings @ hyp_embedding
    ranked = sorted(zip(corpus_ids, scores.tolist()), key=lambda x: x[1], reverse=True)
    return ranked[:top_k], hypothetical


vague_query = "How is my portfolio positioned for a recession?"

print("Test query:", vague_query)
print("=" * 72)

# Standard semantic search
sem_results = semantic_search(vague_query, top_k=3)
sem_ids = [r[0] for r in sem_results]

# HyDE
hyde_results, hypothetical_doc = hyde_retrieve(vague_query, top_k=3)
hyde_ids = [r[0] for r in hyde_results]

print(f"\nHypothetical document generated by HyDE:")
print(f"  '{hypothetical_doc}'")
print(f"\nStandard semantic top-3 : {sem_ids}")
print(f"HyDE top-3              : {hyde_ids}")
print(f"Difference              : {[d for d in hyde_ids if d not in sem_ids]}")

print("\nDocuments retrieved by HyDE:")
id_to_text = {d["id"]: d["text"] for d in corpus}
for doc_id, score in hyde_results:
    print(f"  [{doc_id}] {score:.3f}  {id_to_text[doc_id][:80]}...")

HyDE works because it converts user vocabulary to domain vocabulary automatically. The user says "positioned for a recession" — the hypothetical uses "defensive allocation", "low beta equities", "duration risk" — terms that align with how financial documents are actually written. Cost: one extra API call per query. Worth it for vague or conceptual queries. Not needed for exact ticker queries where standard semantic search already has strong signal.

---
## Section 6 — Query Decomposition

Multi-intent queries produce diffuse embeddings. "How does my tech concentration compare to the S&P 500 and should I rebalance given current interest rates?" is actually three questions. A single embedding has to represent all three intents simultaneously and ends up representing none of them precisely.

In [ ]:
def decompose_query(query: str) -> list[str]:
    raw = client.messages.create(
        model=MODEL,
        max_tokens=300,
        system=(
            "Break this financial question into 2-4 specific sub-questions that can each "
            "be answered independently from a different data source. "
            "Return JSON only — no markdown: {\"sub_queries\": [list of strings]}"
        ),
        messages=[{"role": "user", "content": query}]
    ).content[0].text.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    return json.loads(raw)["sub_queries"]


multi_intent_query = (
    "How does my tech concentration compare to the S&P 500 "
    "and should I rebalance given current interest rates?"
)

print("Multi-intent query:")
print(f"  '{multi_intent_query}'")
print()

# Single retrieval
single_results = hybrid_search(multi_intent_query, top_k=3)
single_ids = [r[0] for r in single_results]
print(f"Single query — hybrid top-3: {single_ids}")

# Decomposed retrieval
sub_queries = decompose_query(multi_intent_query)
print(f"\nDecomposed into {len(sub_queries)} sub-queries:")
all_sub_ids = set()
for i, sq in enumerate(sub_queries, 1):
    sub_results = hybrid_search(sq, top_k=3)
    sub_ids = [r[0] for r in sub_results]
    all_sub_ids.update(sub_ids)
    print(f"  {i}. '{sq}'")
    print(f"     Retrieved: {sub_ids}")

print(f"\nSingle query coverage  : {single_ids}")
print(f"Decomposed coverage    : {sorted(all_sub_ids)}")
print(f"Additional docs found  : {sorted(all_sub_ids - set(single_ids))}")

Multi-intent queries produce diffuse embeddings that miss specific chunks for each intent. Decomposition costs one extra Claude call but retrieves with surgical precision — each sub-query gets an embedding that represents a single intent. In FinMentor, almost every user question is multi-intent: "how am I doing" contains portfolio performance, sector concentration, and benchmark comparison all at once.

---
## Section 7 — The Complete Production Pipeline

Combining all techniques into one function that routes each query through the appropriate retrieval strategy.

In [ ]:
import re

TICKER_PATTERN = re.compile(r'\b[A-Z]{2,5}\b')

def is_multi_intent(query: str) -> bool:
    return " and " in query.lower() or query.count("?") > 1

def is_exact_query(query: str) -> bool:
    return bool(TICKER_PATTERN.search(query))

def is_vague_query(query: str) -> bool:
    specific_terms = ["$", "%", "price", "shares", "earnings", "p&l", "pnl",
                      "allocation", "position"]
    return not any(t in query.lower() for t in specific_terms) and not is_exact_query(query)

def deduplicate(results: list[tuple]) -> list[tuple]:
    seen = set()
    unique = []
    for doc_id, score in results:
        if doc_id not in seen:
            seen.add(doc_id)
            unique.append((doc_id, score))
    return unique


def production_retrieve(query: str, top_k: int = 5) -> tuple[list[tuple], dict]:
    trace = {"query": query, "path": [], "sub_queries": []}

    # Step 1: decompose if multi-intent
    if is_multi_intent(query):
        sub_queries = decompose_query(query)
        trace["path"].append("decomposed")
        trace["sub_queries"] = sub_queries
    else:
        sub_queries = [query]

    all_results = []
    for q in sub_queries:
        if is_exact_query(q):
            # Exact ticker queries: BM25 handles symbol matching precisely
            results = bm25_search(q, top_k=top_k)
            trace["path"].append(f"bm25({q[:30]})")
        elif is_vague_query(q):
            # Vague conceptual queries: HyDE bridges vocabulary gap
            results, hyp = hyde_retrieve(q, top_k=top_k)
            trace["path"].append(f"hyde({q[:30]})")
        else:
            # Everything else: hybrid search
            results = hybrid_search(q, top_k=top_k)
            trace["path"].append(f"hybrid({q[:30]})")
        all_results.extend(results)

    # Step 2: deduplicate and rerank
    unique = deduplicate(all_results)
    candidate_ids = [r[0] for r in unique[:10]]
    final = rerank(query, candidate_ids, top_n=top_k)
    trace["path"].append("reranked")
    return final, trace


test_cases = [
    "NVDA position",
    "am I positioned for a downturn?",
    "compare my tech vs S&P and should I rebalance?"
]

id_to_text = {d["id"]: d["text"] for d in corpus}

print("=" * 72)
print("PRODUCTION PIPELINE — per-query routing")
print("=" * 72)
for q in test_cases:
    final, trace = production_retrieve(q, top_k=3)
    print(f"\nQ: {q}")
    print(f"   Path      : {' → '.join(trace['path'])}")
    if trace["sub_queries"]:
        for i, sq in enumerate(trace["sub_queries"], 1):
            print(f"   sub-q {i}   : {sq}")
    print(f"   Final top-{len(final)}:")
    for doc_id, score in final:
        print(f"     [{doc_id}] {score:+.4f}  {id_to_text[doc_id][:70]}...")

---
## Section 8 — Key Observations

1. **Vocabulary mismatch is the most common RAG failure in financial and enterprise domains.** Hybrid search is not optional — it's the baseline. Pure semantic search systematically misses formal/informal vocabulary variants that are ubiquitous in financial text.

2. **RRF is elegant precisely because it ignores raw scores.** Cosine similarity and BM25 scores are on different scales and can't be combined directly without calibration. RRF uses only rank position — a document appearing in both lists gets a natural boost without any normalization required.

3. **HyDE's hypothetical document is more valuable than it sounds.** It automatically converts casual user language to formal domain vocabulary. The user says "positioned for a downturn" — the hypothetical uses "defensive allocation", "low beta", "duration risk" — terms that appear in the indexed documents. Cost: one extra API call. Worth it for vague queries.

4. **Query decomposition is the highest-leverage improvement for complex questions.** One extra Claude call per multi-intent query, dramatically better coverage. In FinMentor, almost every financial question is multi-intent — "how am I doing" embeds portfolio performance, sector concentration, and benchmark comparison simultaneously.

5. **The debugging sequence when RAG gives wrong answers:** generation → retrieved chunks → index existence → query routing. Always start from the output and work backward. Most "Claude is wrong" bugs are actually retrieval bugs visible the moment you print what was retrieved.

In [ ]:
print("Commit message:")
print("  04B: advanced retrieval — hybrid search, BM25, RRF, cross-encoder reranking, HyDE, query decomposition")
print()
print("Push command:")
print("  git add 04-rag-systems/")
print("  git commit -m '04B: advanced retrieval — hybrid search, BM25, RRF, cross-encoder reranking, HyDE, query decomposition'")
print("  git push origin main")